# 04_genai.ipynb: Patient Call-Centre Note Classification
This notebook loads unstructured qualitative patient follow-up phone call transcripts, parses them into structured insights using Pydantic, and validates them under defined clinic standards.

In [ ]:
import os
import pandas as pd
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List
import json

## Define Pydantic Models for Structured Output Validation

In [ ]:
ALLOWED_ROOT_CAUSES = [
    'forgot_or_no_reminder',
    'distance_or_transport',
    'cost_or_fee',
    'illness_or_emergency',
    'work_or_schedule',
    'other'
]

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator('root_cause')
    def check_allowed_root_cause(cls, v):
        if v not in ALLOWED_ROOT_CAUSES:
            raise ValueError(f"root_cause must be one of {ALLOWED_ROOT_CAUSES}")
        return v

## Simulation of Gemini API Structure Parsing
This block runs the unstructured text classification and structures the validated results.

In [ ]:
# Mock / local classification function imitating the Gemini parser output
def local_fallback_classifier(text):
    text_lower = text.lower()
    if 'forget' in text_lower or 'reminder' in text_lower or 'bhool' in text_lower:
        return {
            'root_cause': 'forgot_or_no_reminder',
            'confidence': 0.90,
            'summary': 'Forgot appointment or missed text reminder',
            'recommended_action': 'Optimize automated SMS reminders'
        }
    elif 'far' in text_lower or 'transport' in text_lower:
        return {
            'root_cause': 'distance_or_transport',
            'confidence': 0.95,
            'summary': 'Distance was too far / lack of transit options',
            'recommended_action': 'Evaluate and suggest telemedicine consultation alternative'
        }
    elif 'fee' in text_lower or 'cost' in text_lower or 'afford' in text_lower:
        return {
            'root_cause': 'cost_or_fee',
            'confidence': 0.90,
            'summary': 'Clinic fees are prohibitively expensive',
            'recommended_action': 'Suggest flexible payments/subsidies'
        }
    elif 'emergency' in text_lower or 'illness' in text_lower or 'suffering' in text_lower:
        return {
            'root_cause': 'illness_or_emergency',
            'confidence': 0.95,
            'summary': 'Emergency or medical issues',
            'recommended_action': 'Proactively reschedule'
        }
    elif 'busy' in text_lower or 'work' in text_lower or 'office' in text_lower:
        return {
            'root_cause': 'work_or_schedule',
            'confidence': 0.90,
            'summary': 'Conflicting professional work schedule',
            'recommended_action': 'Offer evening/weekend slots'
        }
    else:
        return {
            'root_cause': 'other',
            'confidence': 0.50,
            'summary': 'Other unclassified reasons',
            'recommended_action': 'Follow up with a second call'
        }

In [ ]:
notes_df = pd.read_csv('data/followup_notes.csv')
results = []

for idx, row in notes_df.iterrows():
    note_id = row['note_id']
    appt_id = row['appointment_id']
    text = row['note_text']

    if pd.isna(text) or str(text).strip() == '':
        res = {
            'note_id': note_id,
            'appointment_id': appt_id,
            'root_cause': 'other',
            'confidence': 0.0,
            'summary': 'No note recorded',
            'recommended_action': 'Re-call patient',
            'validation_status': 'skipped_blank'
        }
    else:
        pred = local_fallback_classifier(text)
        try:
            validated = FollowupInsight(**pred)
            status = 'valid'
        except ValidationError:
            status = 'invalid'

        res = {
            'note_id': note_id,
            'appointment_id': appt_id,
            'root_cause': pred['root_cause'],
            'confidence': pred['confidence'],
            'summary': pred['summary'],
            'recommended_action': pred['recommended_action'],
            'validation_status': status
        }
    results.append(res)

analysed_df = pd.DataFrame(results)
display(analysed_df)